In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.5 MB/s eta 0:00:00


In [ ]:
import os
import shutil
import yaml
import numpy as np
import torch
from ultralytics import YOLO
from datetime import datetime
import cv2
import matplotlib.pyplot as plt

# Definir o diretório base para o projeto
base_dir = 'content/drive/MyDriveyolov8_dataset'

# Criar diretórios principais se não existirem
os.makedirs(base_dir, exist_ok=True)

# Definir os caminhos para os dados de treino, validação e teste
train_path = os.path.join(base_dir, 'train')
val_path = os.path.join(base_dir, 'val')
test_path = os.path.join(base_dir, 'test')

# Criar subdiretórios 'images' e 'labels' dentro de treino, validação e teste
os.makedirs(os.path.join(train_path, 'images'), exist_ok=True)
os.makedirs(os.path.join(train_path, 'labels'), exist_ok=True)
os.makedirs(os.path.join(val_path, 'images'), exist_ok=True)
os.makedirs(os.path.join(val_path, 'labels'), exist_ok=True)
os.makedirs(os.path.join(test_path, 'images'), exist_ok=True)
os.makedirs(os.path.join(test_path, 'labels'), exist_ok=True)

# Definir o caminho para o arquivo data.yaml
data_yaml_path = os.path.join(base_dir, 'data.yaml')

# Definir os nomes das classes (substitua com suas classes reais)
class_names = ['caneca', 'joystic', 'mouse', 'outros']

# Definir o diretório para salvar os modelos
model_save_dir = os.path.join(base_dir, 'models')
os.makedirs(model_save_dir, exist_ok=True)

print(f"Estrutura de pastas e variáveis globais configuradas em: {base_dir}")
print("Por favor, popule as pastas 'images' e 'labels' com seus respectivos dados.")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Estrutura de pastas e variáveis globais configuradas em: /content/yolov8_dataset
Por favor, popule as pastas 'images' e 'labels' com seus respectivos dados.


# Dataset Verification

In [3]:
def verify_dataset_structure():
    # Check if data.yaml exists and create if needed
    if not os.path.exists(data_yaml_path):
        train_images_path = os.path.join(train_path, "images")
        val_images_path = os.path.join(val_path, "images")
        test_images_path = os.path.join(test_path, "images")

        yaml_data = {
            'train': train_images_path,
            'val': val_images_path,
            'test': test_images_path,
            'nc': len(class_names),
            'names': class_names
        }

        with open(data_yaml_path, 'w') as f:
            yaml.dump(yaml_data, f, default_flow_style=False)

    # Count and verify images and labels
    train_images_dir = os.path.join(train_path, "images")
    train_labels_dir = os.path.join(train_path, "labels")

    train_images = len([f for f in os.listdir(train_images_dir)
                       if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
    train_labels = len([f for f in os.listdir(train_labels_dir) if f.endswith('.txt')])

    return train_images > 0 and train_labels > 0

# Creating a Validation Set

In [4]:
def create_validation_set(train_images_dir, train_labels_dir, val_images_dir, val_labels_dir):
    os.makedirs(val_images_dir, exist_ok=True)
    os.makedirs(val_labels_dir, exist_ok=True)

    image_files = [f for f in os.listdir(train_images_dir)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]

    np.random.seed(42)
    np.random.shuffle(image_files)
    val_split = int(0.2 * len(image_files))
    val_files = image_files[:val_split]

    for img_file in val_files:
        # Copy image
        src_img = os.path.join(train_images_dir, img_file)
        dst_img = os.path.join(val_images_dir, img_file)
        shutil.copy2(src_img, dst_img)

        # Copy corresponding label file
        label_file = os.path.splitext(img_file)[0] + '.txt'
        src_label = os.path.join(train_labels_dir, label_file)
        dst_label = os.path.join(val_labels_dir, label_file)

        if os.path.exists(src_label):
            shutil.copy2(src_label, dst_label)

# Training the Model

In [5]:
def train_yolo_model(epochs=50, batch_size=16, img_size=640, lr0=0.01):
    # Check for CUDA availability
    device = '0' if torch.cuda.is_available() else 'cpu'

    # Define timestamp for unique model naming
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = f'train_{timestamp}'

    # Load the model
    try:
        model = YOLO('yolov5n.pt') # Prioritizing yolov5n for smaller datasets
        model_type = 'yolov5n'
    except Exception:
        try:
            model = YOLO('yolo12n.pt')
            model_type = 'yolo12n'
        except Exception:
            model = YOLO('yolov8n.pt')
            model_type = 'yolov8n'
    # model = YOLO('yolov5n.pt')
    # model_type = 'yolov5n'


    # Train the model
    results = model.train(
        data=data_yaml_path,
        epochs=epochs,
        batch=batch_size,
        imgsz=img_size,
        patience=10,
        save=True,
        device=device,
        project=os.path.join(base_dir, 'runs'),
        name=run_name,
        lr0=lr0,
        lrf=0.01,
        plots=True,
        save_period=5,

        # AUGMENTATION OTIMIZADO
        hsv_h=0.02,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=8.0,
        translate=0.15,
        scale=0.6,
        shear=2.0,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
    )

    # Save the model
    # model_save_path = os.path.join(model_save_dir, f"{model_type}_{timestamp}.pt")

    # try:
    #     model.model.save(model_save_path)
    # except AttributeError:
    #     try:
    #         model.save(model_save_path)
    #     except Exception:
    #         best_model_path = os.path.join(base_dir, 'runs', run_name, 'weights', 'best.pt')
    #         if os.path.exists(best_model_path):
    #             shutil.copy2(best_model_path, model_save_path)
    """
    O bloco de código 'Save the model' estava retornando:
    - TypeError: 'list' object is not callable
    Troquei para um método alternativo de salvar o modelo, que funcione em
    versões mais modernas do YOLO.
    """

    # Save the model
    model_save_path = os.path.join(model_save_dir, f"{model_type}_{timestamp}.pt")

    best_model_path = os.path.join(base_dir, 'runs', run_name, 'weights', 'best.pt')

    if os.path.exists(best_model_path):
        shutil.copy2(best_model_path, model_save_path)
    else:
        model.save(model_save_path)


    return model

# Model Validation

In [6]:
def validate_model(model):
    metrics = model.val(
        data=data_yaml_path,
        split='val',
        project=os.path.join(base_dir, 'runs'),
        name='val'
    )

    # Calculate F1 score
    f1_score = 2 * metrics.box.mp * metrics.box.mr / (metrics.box.mp + metrics.box.mr + 1e-6)

    return metrics

# Testing on Images

In [7]:
def test_on_images(model, conf_threshold=0.05):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = os.path.join(base_dir, 'runs', f'detect_{timestamp}')
    os.makedirs(output_dir, exist_ok=True)

    test_images_dir = os.path.join(test_path, "images")
    image_files = [f for f in os.listdir(test_images_dir)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]

    with open(data_yaml_path, 'r') as f:
        yaml_data = yaml.safe_load(f)

    class_names = yaml_data.get('names', ['Unknown'])

    # Process a subset of images for visualization
    viz_images = image_files[:min(10, len(image_files))]

    for img_file in viz_images:
        img_path = os.path.join(test_images_dir, img_file)

        # Run inference
        results = model(img_path, conf=conf_threshold)

        # Get original image for visualization
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Draw boxes on image
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            conf = box.conf[0].item()
            cls = int(box.cls[0].item())

            # Get class name
            class_name = class_names[cls] if cls < len(class_names) else f"Unknown-{cls}"

            # Generate a unique color for each class
            color = ((cls * 70) % 256, (cls * 50) % 256, (cls * 30) % 256)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

            # Add label
            label = f"{class_name}: {conf:.2f}"
            cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # Save annotated image
        output_path = os.path.join(output_dir, img_file)
        plt.figure(figsize=(12, 12))
        plt.imshow(img)
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(output_path)
        plt.close()

# Main Function

In [ ]:
def main():
    # Verify dataset structure
    dataset_valid = verify_dataset_structure()
    if not dataset_valid:
        return

    # Define timestamp for this training run
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Train model
    model = train_yolo_model(epochs=100, batch_size=16, lr0=0.001)

    if model is not None:
        validate_model(model)
        test_on_images(model)

In [ ]:
if __name__ == "__main__":
    main()

PRO TIP 💡 Replace 'model=yolov5n.pt' with new 'model=yolov5nu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.

Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolov8_dataset/data.yaml, degrees=8.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=

Essa mudança gerou uma queda na Recall, mas acho que influenciou nos (no detections) que surgiram, talvez a maneira como foi ajustada o dataset pode ter influenciado negativo nessa parte.

Obs: eu tentei nos dois modelos do YOLO, inclusive no que deu certo da ultima vez